# ヒューマン・イン・ザ・ループ：事前アクションゲート、リスク階層化、および監査ログ

このレッスンのREADMEでは、エージェントがすでに応答を生成した後でユーザーに `APPROVE` か `REJECT` を尋ねる短いスニペットでヒューマン・イン・ザ・ループを紹介しています。そのパターンは良い出発点ですが、実際のHITL実装では通常、さらに3つの要素が必要です：

1. エージェントがリスクのあるステップを実行する<strong>前に</strong>動作する<strong>事前アクションゲート</strong>。これによりコスト、取り返しのつかない操作、およびレイテンシを制御できます。
2. <strong>リスク階層化</strong>。低リスクのアクションは自動実行し、中リスクはバッチ承認し、高リスクのみ人間の承認を必要とします。
3. <strong>監査ログと改訂ループ</strong>。すべてのゲート決定をJSONLとして記録し、拒否の場合は単に `Revising...` と表示するのではなく、構造化された理由とともにエージェントに再プロンプトを送ります。

このノートブックは `06-system-message-framework.ipynb` と同じプリミティブ上にこれらすべてを構築しています。`DEMO_MODE = True` でエンドツーエンドに実行（対話入力は不要）、`DEMO_MODE = False` では実際に `input()` プロンプトを使います。注意：DEMO_MODEでは3つ目の目標のリトライがスクリプト化されているため、ループのメカニクスをエンドツーエンドで確認できます。実際の改訂駆動の再分類は `DEMO_MODE = False` とオペレーターが必要です。

**範囲外（他のレッスンで扱う）：** 認証とアクセス制御（レッスン06 READMEの脅威2）、ツールコールミドルウェア（レッスン14 MAF詳細解説）、マルチエージェント討論パターン。


In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

# This notebook uses the Azure OpenAI Responses API via the stable /openai/v1/ endpoint.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API.
endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
if not endpoint:
    raise RuntimeError(
        "AZURE_OPENAI_ENDPOINT environment variable is not set. This notebook needs "
        "an Azure OpenAI resource with a model deployment that supports the Responses "
        "API. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_DEPLOYMENT in "
        "your environment or a local .env file, then run `az login`."
    )

deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# Authenticate with Entra ID (run `az login` first). No api_version is needed.
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

client = OpenAI(
    base_url=f"{endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)



## パターン1: 事前アクションゲート

READMEのHITLスニペットはまずエージェントを呼び出し、その後でユーザーに出力の承認を求めます。これは<strong>事後アクション</strong>のフローです。エージェントはすでに実行されているため、LLM呼び出しコストはすでに支払われており、いかなる副作用（メール送信、データベース行の書き込み、コメント投稿）も既に発生しています。

<strong>事前アクション</strong>のフローは、リスクのあるステップをエージェントが実行する前にゲートを挿入します。エージェントがアクションを提案し、ゲートが実行の可否を決定し、承認された場合にのみ副作用が発生します。

| 観点 | 事後承認 (READMEスニペット) | 事前アクションゲート (このノートブック) |
|---|---|---|
| 承認はいつ行われるか？ | エージェントが出力を生成した後 | 副作用が実行される前 |
| 却下時のLLMコスト | 既に支払われている | 提案に対してのみ支払われ、アクションには支払われない |
| 取り消せない副作用 | 可能（アクションはすでに起きている） | 防止される |
| 監査の明確さ | 承認はprint文 | 承認はタイムスタンプ、アクション、理由を含むJSONレコード |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## パターン 2: リスク階層化

すべての操作が人間の承認を必要とするわけではありません。パブリックAPIへの読み取り専用のルックアップは、顧客へのメール送信とは異なるリスクがあります。両方を同じように扱うと、オペレーターの注意が無駄になり、エージェントの動作を遅くします。

シンプルな3段階モデル：

| 階層 | 例 | 承認フロー |
|---|---|---|
| `low` (読み取り専用) | ナレッジベースを検索する、フライトオプションを調べる、パブリックウェブページを取得する | 自動実行、監査のために記録 |
| `medium` (軽微な変更) | 結果をキャッシュする、カウンターを増加させる、リマインダーをスケジュールする | 自動実行だが、日次レビューで一括処理 |
| `high` (外部向けまたは取り消し不可能) | メールを送信する、カードに請求する、公開チャンネルに投稿する | 人間の承認でブロック |

これはひとつの階層化例です。実稼働システムでは、より細かい階層（例：AWS IAMの権限レベル、ロールベースアクセス階層）を使うことが多いです。下記の3段階バージョンは、読み取り専用と副作用を持つ操作が混在するエージェントにとって最小限の有用なバージョンです。

以下の分類器はキーワードのヒューリスティックを使用しており、デモが決定的かつ安価に保たれるようにしています。実稼働システムでは、これを学習済み分類器やポリシーエンジンに置き換えることになります。


In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## パターン3: 監査ログと改訂ループ

`print("Response approved.")` は監査ログではありません。信頼性のために、すべてのゲート決定は、後でクエリ、再生、またはインシデントレビューに添付できる構造化イベントとして記録されるべきです。

2つのポイント：

1. **追加のみのJSONL。** 1行に1つの決定を、タイムスタンプ、アクション、階層、決定、理由とともに記録。grepが簡単で、後で実際のログストアに送るのも簡単です。
2. **拒否時の改訂ループ。** ゲートが `deny` を返すと、エージェントは拒否理由をコンテキストに含めて自分自身に再度プロンプトを送り、次の提案が問題を回避できるようにします。


In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.responses.create(
        model=deployment,
        input=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_text},
        ],
        store=False,
    )
    return response.output_text.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}



In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## 追加リソース

これらのHITLパターンのバリエーションを実装している他のいくつかの公開プロジェクトもあります。アプローチを比較して、ご自身の環境に合うものを見つけてください:

- **LangChain** human-in-the-loop ツールラッパー（[ドキュメント](https://python.langchain.com/docs/integrations/tools/human_tools)）：人間の入力のために実行を一時停止するドロップインツールラッパー。
- **AutoGen** `UserProxyAgent`（[v0.2 ドキュメント](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+では構造が変更されています）：マルチエージェント会話で人間を表すためのエージェントロールを使用。
- **Microsoft Agent Framework (MAF)** 関数呼び出しミドルウェア（[ドキュメント](https://learn.microsoft.com/agent-framework/)）：すべてのツール/関数呼び出しの前後で動作するミドルウェアで、ゲーティングロジックや承認フローに適しています。

各プロジェクトは3つのサブパターンを異なる方法で扱っています：LangChainはツールとしてラップし、AutoGenはエージェントロールを使用し、Microsoft Agent Frameworkは関数呼び出しミドルウェアを使用します。ご自身のエージェントの設計を決める前に、1つか2つの実装を通しで読んでください。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責事項**：
本書類は AI 翻訳サービス [Co-op Translator](https://github.com/Azure/co-op-translator) を使用して翻訳されています。正確性を期していますが、自動翻訳には誤りや不正確な部分が含まれる可能性があることをご承知おきください。原文の原語版が正式な情報源とみなされるべきです。重要な情報については、専門の人間による翻訳を推奨します。本翻訳の利用により生じたいかなる誤解や解釈違いについても、当方は責任を負いかねます。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
